# R4-N1 BiGRU-A calibration (Protocol A)

This notebook runs the project’s shared R4-N1 implementation for **BiGRU-A only**. It fits on calibration-role recordings, writes and verifies the frozen fit stage, and only then opens operating predictions. It never selects on or reads the sealed test partition.

The notebook refuses to start if `runs/r4/r4_n1_bigru/` already exists. After reviewing a completed run, archive it elsewhere before intentionally starting a new run. The approved LR-A output and all Protocol B files are outside this workflow.

## 1. Mount Drive and locate the project

Expected project root: `MyDrive/PAI&AIES Project`. The project source, frozen recipe, BiGRU-A checkpoint, training/export evidence, and six role Parquets must already be present there.

In [2]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/PAI&AIES Project').resolve()
CONFIG_PATH = PROJECT_ROOT / 'configs/r4_n1_calibration_a.json'
RUN_DIR = PROJECT_ROOT / 'runs/r4/r4_n1_bigru'

required_project_files = [
    PROJECT_ROOT / 'src/echoproof/evaluation/r4_n1.py',
    PROJECT_ROOT / 'scripts/r4_n1_calibrate.py',
    CONFIG_PATH,
]
missing = [str(path) for path in required_project_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing project files in Drive:\n' + '\n'.join(missing))
if RUN_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite existing BiGRU-A run: {RUN_DIR}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Calibration recipe: {CONFIG_PATH}')
print(f'New output directory: {RUN_DIR}')
print('Existing run check: clear; no run was created yet.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project
Calibration recipe: /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/configs/r4_n1_calibration_a.json
New output directory: /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/runs/r4/r4_n1_bigru
Existing run check: clear; no run was created yet.


## 2. Prepare the Python environment

The notebook imports the checked-out project source directly. It installs only packages that are missing; it does not upgrade packages already present, and it does not install model-training dependencies because calibration consumes the frozen predictions rather than loading the neural network.

In [3]:
import importlib.util
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError(f'EchoProof requires Python 3.11 or newer; Colab has {sys.version.split()[0]}')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
required_modules = {
    'numpy': 'numpy',
    'scipy': 'scipy',
    'pyarrow': 'pyarrow',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
}
missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages only:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing_packages])
else:
    print('All calibration dependencies are already available; no package changes needed.')

from echoproof.evaluation import r4_n1
from echoproof.artifacts import file_digest

print('Imported project module:', r4_n1.__file__)
print('Python:', sys.version.replace('\n', ' '))

All calibration dependencies are already available; no package changes needed.
Imported project module: /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/src/echoproof/evaluation/r4_n1.py
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


## 3. Verify the frozen BiGRU-A inputs before fitting

This preflight verifies the checkpoint, training evidence, export evidence, selection and calibration recording/chunk Parquet hashes, schemas, role counts, component labels/counts, recording/chunk parity, and selection/calibration speaker separation. It checks only that operating files exist and are listed in the signed-by-hash export evidence; it does **not** hash or read their contents in this cell. The R4-S1 manifest hash is checked as an evidence binding; the manifest itself is not independently verified unless available in the project checkout.

In [4]:
recipe = r4_n1.load_recipe(CONFIG_PATH)
if recipe['models']['bigru']['verification_state'] != 'verified_local':
    raise RuntimeError('Recipe does not mark BiGRU-A inputs as verified.')
audit = r4_n1.audit_inputs(PROJECT_ROOT, recipe, model_name='bigru')
print('Selection/calibration preflight checks:')
for check, passed in audit.get('checks', {}).items():
    print(f"  {'PASS' if passed else 'FAIL'}  {check}")
if not audit.get('passed'):
    raise RuntimeError('BiGRU-A selection/calibration input audit failed:\n' + '\n'.join(audit.get('problems', [])))
print('Validated role summaries:', audit['roles'])
print('Recording/chunk mean-score maximum absolute differences:', audit['chunk_mean_max_abs_difference'])
print('Operating prediction contents read:', audit['operating_predictions_content_read'])
print('R4-S1 manifest:', audit['r4_s1_manifest_verification'])
print('Preflight passed. No model fitting has been run in this cell.')

Selection/calibration preflight checks:
  PASS  training_evidence_sha256
  PASS  export_evidence_sha256
  PASS  selected_model_sha256
  PASS  training_complete
  PASS  export_complete
  PASS  split_hash
  PASS  training_evidence_link
  PASS  selected_model_hash
  PASS  r4_s1_manifest_hash
  PASS  export_hash_map
  PASS  prediction_sha256:predictions_chunks_selection.parquet
  PASS  prediction_sha256:predictions_chunks_calibration.parquet
  PASS  operating_file_present:predictions_chunks_operating.parquet
  PASS  prediction_sha256:predictions_recordings_selection.parquet
  PASS  prediction_sha256:predictions_recordings_calibration.parquet
  PASS  operating_file_present:predictions_recordings_operating.parquet
  PASS  selection_calibration_speakers_disjoint
  PASS  selection_calibration_recordings_disjoint
Validated role summaries: {'selection': {'recordings': 2500, 'speakers': 25}, 'calibration': {'recordings': 1200, 'speakers': 12}}
Recording/chunk mean-score maximum absolute differenc

## 4. Run BiGRU-A calibration

The project runner repeats the verified preflight, fits identity/temperature/affine candidates on calibration recordings only, performs speaker-held-out LOSO comparisons, selects and freezes the mapping under the predeclared one-standard-error rule, and persists the fit evidence. The operating stage is invoked only after the fit completion record and its file hashes validate. It then verifies the operating Parquets and performs the predeclared acceptance and threshold procedure. This cell is the only cell that performs the actual calibration.

In [5]:
import json
import importlib.metadata
from datetime import datetime, timezone

if RUN_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite existing BiGRU-A run: {RUN_DIR}')
started_utc = datetime.now(timezone.utc).isoformat()
command = [
    sys.executable, str(PROJECT_ROOT / 'scripts/r4_n1_calibrate.py'),
    '--model', 'bigru', '--root', str(PROJECT_ROOT), '--config', str(CONFIG_PATH),
]
print('Starting BiGRU-A calibration using:', ' '.join(command))
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
if completed.returncode != 0:
    partial_run = str(RUN_DIR) if RUN_DIR.exists() else 'no run directory created'
    raise RuntimeError(f"BiGRU-A calibration runner failed with exit code {completed.returncode}; inspect {partial_run}.")
if not (RUN_DIR / 'fit/stage_complete.json').is_file() or not (RUN_DIR / 'operating/stage_complete.json').is_file():
    raise RuntimeError('Runner returned success without both stage completion records.')
completed_utc = datetime.now(timezone.utc).isoformat()
env = {
    'model': 'bigru', 'protocol': 'A', 'python': sys.version, 'platform': sys.platform,
    'started_utc': started_utc, 'completed_utc': completed_utc,
    'packages': {name: importlib.metadata.version(name) for name in ['numpy', 'scipy', 'pyarrow', 'matplotlib', 'scikit-learn']},
    'command': command, 'project_root': str(PROJECT_ROOT), 'recipe_sha256': file_digest(CONFIG_PATH),
    'runner_returncode': completed.returncode, 'test_data_read': False,
}
env_path = RUN_DIR / 'execution_environment.json'
with env_path.open('x', encoding='utf-8') as stream:
    json.dump(env, stream, indent=2, sort_keys=True)
    stream.write('\n')
print(f'BiGRU-A run completed. Execution record: {env_path}')

Starting BiGRU-A calibration using: /usr/bin/python3 /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/scripts/r4_n1_calibrate.py --model bigru --root /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project --config /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/configs/r4_n1_calibration_a.json
{
  "mapping_accepted": false,
  "model": "bigru",
  "protocol": "A",
  "run_dir": "/content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/runs/r4/r4_n1_bigru",
  "selected_mapping": "identity",
  "status": "complete",
  "threshold_rule": {
    "decision": "score >= threshold",
    "kind": "threshold",
    "threshold": 0.610876834796857
  }
}

BiGRU-A run completed. Execution record: /content/drive/.shortcut-targets-by-id/1icE6vOFhCIuhCe6gfhFE2JI2yGHGBbMR/PAI&AIES Project/runs/r4/r4_n1_bigru/execution_environment.json


## 5. Review generated evidence

The report below reads only R4-N1 run outputs. The run contains input audit evidence, LOSO and final-fit evidence, a mapping-freeze completion record, and operating validation/results with the reliability plot, component diagnostics, speaker diagnostics, bootstrap intervals, threshold plateau counts, and completion record. No test partition is accessed.

In [8]:
fit_report = json.loads((RUN_DIR / 'fit/calibration.json').read_text(encoding='utf-8'))
loso_report = json.loads((RUN_DIR / 'fit/loso_comparison.json').read_text(encoding='utf-8'))
operating_report = json.loads((RUN_DIR / 'operating/operating_results.json').read_text(encoding='utf-8'))
print('Selected mapping:', fit_report['selected_mapping'])
print('Selected parameters:', fit_report['selected_parameters'])
print('LOSO means:', fit_report['selection']['means'])
print('Excluded candidates:', fit_report['selection']['excluded'])
print('Calibration accepted:', operating_report['mapping_accepted'])
print('Deployed score type:', operating_report['score_type'])
print('Threshold:', operating_report['threshold_rule'])
print('Threshold plateau:', operating_report['threshold_metrics']['plateau'])
print('Operating probability metrics:')
for view, metrics in operating_report['probability_metrics'].items():
    print(f"  {view}: ROC-AUC={metrics['roc_auc']:.6f}; EER={metrics['eer']:.6f}; LogLoss={metrics['log_loss']:.6f}; Brier={metrics['brier']:.6f}; ECE={metrics['ece']:.6f}")
print('Selected-threshold macro F1:', operating_report['threshold_metrics']['macro_f1'])
print('Per-generator diagnostics:', sorted(operating_report['per_component']))
print('Per-speaker diagnostic rows:', len(operating_report['per_speaker_false_alarms']))
print('Speaker bootstrap:', operating_report['speaker_bootstrap'])
print('Reliability plot:', RUN_DIR / operating_report['reliability_plot']['path'])
print('All run artifacts:')
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to(RUN_DIR)} ({path.stat().st_size} bytes)')

Selected mapping: identity
Selected parameters: {}
LOSO means: {'affine': 0.024716905224529764, 'identity': 0.02409021057895798, 'temperature': 0.03469454469234291}
Excluded candidates: {}
Calibration accepted: False
Deployed score type: model_score
Threshold: {'decision': 'score >= threshold', 'kind': 'threshold', 'threshold': 0.610876834796857}
Threshold plateau: {'all_negative_tied_at_max_macro_f1': False, 'macro_f1_max': 0.9741180954331237, 'thresholds_remaining_after_closeness_tie_break': 1, 'thresholds_remaining_after_fpr_tie_break': 1, 'thresholds_tied_at_max_macro_f1': 1, 'tolerance': 1e-12}
Operating probability metrics:
  deployed: ROC-AUC=0.980671; EER=0.050000; LogLoss=0.124276; Brier=0.015873; ECE=0.016934
  mapped: ROC-AUC=0.980671; EER=0.050000; LogLoss=0.124276; Brier=0.015873; ECE=0.016934
  raw: ROC-AUC=0.980671; EER=0.050000; LogLoss=0.124276; Brier=0.015873; ECE=0.016934
Selected-threshold macro F1: 0.9741180954331237
Per-generator diagnostics: ['chatterbox', 'melo'